Imports

In [8]:
import sys
!{sys.executable} -m pip install -r ./requirements.txt

  Using cached jaal-0.1.9-py3-none-any.whl.metadata (7.5 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached dash-3.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached visdcc-0.0.63-py3-none-any.whl.metadata (15 kB)
  Using cached dash_core_components-2.0.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached dash_html_components-2.0.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached dash_bootstrap_components-0.13.1-py3-none-any.whl.metadata (5.3 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached flask-3.1.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached werkzeug-3.1.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached plotly-6.2.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached importlib_metadata-8.7.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached retrying-1.4.2-py3-none-any.wh

In [11]:
from pyarn import lockfile
import json

def openFile():
    pathfile = "./yarn.lock"
    my_lockfile = lockfile.Lockfile.from_file(pathfile)
    return my_lockfile.data


def loadRequirements():
    pathfile = './packages.json'
    with open(pathfile, 'r') as f:
        jsonfile = json.load(f)
    accepted_keys = ["dependencies", "devDependencies", "peerDependencies"]
    def filterKeys(pair):
        key, value = pair
        return key in accepted_keys
    return dict(filter(filterKeys, jsonfile.items()))

data = openFile()
requirements = loadRequirements()

In [35]:
# Get the latest version for each package

import requests
import time

def getCurrentVersion(data, name):
    if ('resolved' not in data[name].keys()):
        return ''
    url = data[name]['resolved'].split('/-/')[0]
    resp = requests.get(url)
    package_data = json.loads(resp.content)
    if ('dist-tags' in package_data.keys()):
        return package_data['dist-tags']['latest']
    return ''


def versionnedPackages(data, requirements):
    # dependencies->names->version
    
    # get all versions by level for each package
    aggregated_names = {}
    for level in requirements.keys():
        for req in requirements[level].keys():
            if (req not in aggregated_names):
                aggregated_names[req] = {}
            aggregated_names[req][level] = requirements[level][req]
    unique_filtered_requirements = aggregated_names.keys()
    # names->dependencies->version

    # we find all matching packages in data (listing installed versions)
    matching_packages = {}
    for requirement in unique_filtered_requirements:
        for level in aggregated_names[requirement]:
            if (requirement not in matching_packages.keys()):
                matching_packages[requirement] = []
            # double set to ensure no double in final array and to use filter as array
            matching_packages[requirement] += list(set(filter(lambda x: f"{requirement}@{aggregated_names[requirement][level]}" in x, data.keys())))
        matching_packages[requirement] = list(set(matching_packages[requirement]))
    # react-dom -> [react-dom@1, react-dom@2]
    packages = {}
    #ex react-dom 
    for package in matching_packages.keys():
        print(package)
        latest = getCurrentVersion(data, matching_packages[package][0]) # should be the same resolver for all
        time.sleep(0.1)
        for one_version in matching_packages[package]: # loop on react-dom@1, react-dom@2
            at_level = [] # list all levels in which this version is found
            for level in aggregated_names[package]:
                if (f"{package}@{aggregated_names[package][level]}" in one_version):
                    at_level.append(level)
            for level in at_level:
                if (package in packages.keys()):
                    packages[package]['version'][level] = data[one_version]['version']
                else:
                    packages[package] = {'version': {}, 'latest': latest, 'target': aggregated_names[package]}
                    packages[package]['version'][level] = data[one_version]['version']

    return packages

v_packages = versionnedPackages(data, requirements)

@babel/cli
@babel/core
@babel/helper-call-delegate
@babel/plugin-proposal-class-properties
@babel/plugin-proposal-export-default-from
@babel/plugin-proposal-object-rest-spread
@babel/plugin-syntax-dynamic-import
@babel/plugin-transform-runtime
@babel/preset-env
@babel/preset-react
@storybook/addon-actions
@storybook/addon-docs
@storybook/addon-info
@storybook/addon-knobs
@storybook/addon-notes
@storybook/addon-storysource
@storybook/addon-viewport
@storybook/react
@testing-library/react-hooks
babel-core
babel-eslint
babel-jest
babel-loader
clean-webpack-plugin
coveralls
css-loader
enzyme
enzyme-adapter-react-16
eslint
eslint-config-prettier
eslint-config-react-app
eslint-import-resolver-webpack
eslint-plugin-filenames
eslint-plugin-flowtype
eslint-plugin-import
eslint-plugin-jasmine
eslint-plugin-jest
eslint-plugin-jsx-a11y
eslint-plugin-prettier
eslint-plugin-react
eslint-plugin-react-hooks
file-loader
jest
jest-css-modules
jest-junit
mixpanel-browser
prettier
react
react-color
react-

In [39]:
# Make an excel file for each dependency object with it's packages (target version, current version, latest version)
def saveExcel_versions(listRequirements, listName, packages):
    import pandas as pd
    table = {}
    list_requirements = listRequirements.keys()

    def filterByKey(pair):
        key, value = pair
        return key in list_requirements

    packages_needed = dict(filter(filterByKey, packages.items()))
    attrs = ['version', 'latest', 'target']
    table['package'] = packages_needed.keys()
    for attr in attrs:
        def stepInLevelOptional(item):
            if (type(item[attr]) is dict and 'level' in item[attr].keys()):
                return item[attr][level]
            return item[attr]
        table[attr] = list(map(stepInLevelOptional, packages_needed.values()))
    print('version', len(table['version']))
    print('latest', len(table['latest']))
    print('package', len(table['package']))

    df = pd.DataFrame(table)
    df.to_excel(f"./{listName}.xlsx")

for depLevel in requirements.keys():
    print(depLevel)
    saveExcel_versions(requirements[depLevel], depLevel, v_packages)

devDependencies
version 75
latest 75
package 75
dependencies
version 15
latest 15
package 15
peerDependencies
version 14
latest 14
package 14


Create connections between packages

In [27]:
def mergeVersion(str, data):
    return str + '@' + data['dependencies'][str]


def splitVersion(str):
    return str.rsplit('@', 1)


def makeArray(value):
    if (type(value).__name__ != "list"):
        return [value]
    return value

In [28]:
def detailledConnectPackages(data):
    raw_packages = list(data.keys())
    packages = {}
    connections = []
    for package in raw_packages:
        names = map(lambda x: x.strip(), package.split(','))
        for name in names:
            packages[name] = {'isRoot': False,  'original_name': package}
            if ('dependencies' in data[package].keys()):
                for dep in data[package]['dependencies'].keys():
                    connections_origins = map(lambda x: x[0], connections)
                    connections_targets = map(lambda x: x[1], connections)
                    full_dep = mergeVersion(dep, data[package])

                    if not (name in connections_origins and
                            full_dep in connections_targets):
                        connections.append((name, full_dep))
            # fill attributes for package view
            attributes = ['version', 'dependencies']
            for attribute in attributes:
                if (attribute in data[package].keys()):
                    packages[name][attribute] = data[package][attribute]
    return packages, connections

packages, connections = detailledConnectPackages(data)
print(len(list(packages.keys())))

2739


In [43]:
# other version of the root

# build a revert tree of the dependencies
# taking each dependencies of a package and adding that package on the dependency as a root

def aglomerateAnyRootWithDetail(packages):
    aglomeratedPackages = {}
    
    for package in packages:
        names = package.split(',')
        [namePackage, version] = splitVersion(names[0])
        aglomeratedPackages[namePackage] = {}

        def stepRoot(packages, name, depth, leafcrumb):
            allnames = package.split(',')
            versions = map(lambda x: splitVersion(x)[1], allnames)
            [shortName, version] = splitVersion(allnames[0])

            # prevent infinite loops by keeping track of visited branches
            if (depth < 400 and 'dependencies' in packages[name].keys() and name not in leafcrumb):
                leafcrumb.append(name)
                for dep in packages[name]['dependencies'].keys():
                    if (dep not in aglomeratedPackages.keys()):
                        aglomeratedPackages[dep] = {}
                    full_dep = mergeVersion(dep, packages[name])
                    if ('roots' not in aglomeratedPackages[dep].keys()):
                        aglomeratedPackages[dep]['roots'] = {}
                    if (shortName not in aglomeratedPackages[dep]['roots']):
                        aglomeratedPackages[dep]['roots'][shortName] = []
                    aglomeratedPackages[dep]['roots'][shortName] += versions
                    packages = stepRoot(packages, full_dep, depth + 1, leafcrumb)
            return packages
        packages = stepRoot(packages, package, 0, [])
    return aglomeratedPackages

agglomerated = aglomerateAnyRootWithDetail(packages)

In [29]:
# build a revert tree of the dependencies
# taking each dependencies of a package and adding that package on the dependency as a root

def addRootsOnBranches(packages, name, depth, leafcrumb):
    # prevent infinite loops by keeping track of visited branches
    if (depth < 400 and 'dependencies' in packages[name].keys() and name not in leafcrumb):
        leafcrumb.append(name)
        for dep in packages[name]['dependencies'].keys():
            full_dep = mergeVersion(dep, packages[name])
            if ('roots' not in packages[full_dep].keys()):
                packages[full_dep]['roots'] = []
            if (name not in packages[full_dep]['roots']):
                packages[full_dep]['roots'].append(name)
                packages[full_dep]['roots'].sort()
            packages = addRootsOnBranches(packages, full_dep, depth + 1, leafcrumb)
    return packages

for package in packages:
    packages = addRootsOnBranches(packages, package, 0, [])
for package in packages:
    if ('roots' not in packages[package].keys()):
        packages[package]['isRoot'] = True

In [41]:
def highlightRepetitions(packages, requirements):
    for part in requirements.keys():
        print(part)
        found_included = 0
        for package in requirements[part].keys():
            full_package = package + '@' + requirements[part][package]
            if (full_package in packages.keys()):
                if ('roots' in packages[full_package]):
                    found_included += 1
                    print(f"\t {package} ({requirements[part][package]}) already imported in {packages[full_package]['roots']}")

highlightRepetitions(packages, requirements)

devDependencies
	 @babel/plugin-proposal-class-properties (^7.1.0) already imported in ['jscodeshift@^0.7.0']
	 @babel/plugin-proposal-object-rest-spread (^7.0.0) already imported in ['jscodeshift@^0.7.0']
	 babel-jest (^25.1.0) already imported in ['jest-config@^25.1.0']
	 clean-webpack-plugin (^3.0.0) already imported in ['react-styleguidist@^11.0.1']
	 webpack-merge (^4.2.2) already imported in ['react-styleguidist@^11.0.1']
dependencies
	 prop-types (^15.7.2) already imported in ['@stardust-ui/react-component-event-listener@~0.38.0', '@stardust-ui/react-component-ref@~0.38.0', '@storybook/addon-actions@^5.3.18', '@storybook/addon-docs@^5.3.18', '@storybook/addon-info@^5.3.18', '@storybook/addon-knobs@^5.3.18', '@storybook/addon-notes@^5.3.18', '@storybook/addon-storysource@^5.3.18', '@storybook/addon-viewport@^5.3.18', '@storybook/components@5.3.18', '@storybook/react@^5.3.18', '@storybook/source-loader@5.3.18', '@storybook/theming@5.3.18', '@storybook/ui@5.3.18', 'airbnb-prop-type

In [ ]:
def buildTree(packages):
    final_tree = []
    for package in packages:
        if (packages[package]['isRoot'] and packages[package]['original_name'] not in final_tree):
            final_tree.append(packages[package]['original_name'])
    return final_tree

In [33]:
# run a Jaal browser to display the graph

def makeJaalGraph(packages, connections):
    from jaal import Jaal
    import pandas as pd

    edges_raw = {}
    edges_raw['from'] = map(lambda x: x[0], connections)
    edges_raw['to'] = map(lambda x: x[1], connections)

    nodes_raw = {}
    first_package = list(packages.keys())[0]
    for attribute in packages[first_package].keys():
        def fillAttribute(x):
            if (attribute in x.keys()):
                return x[attribute]
            return None
        nodes_raw[attribute] = map(fillAttribute, packages.values())
    nodes_raw['id'] = list(packages.keys())
    Jaal(pd.DataFrame(edges_raw), pd.DataFrame(nodes_raw)).plot(directed=True)

makeJaalGraph(packages, connections)

Parsing the data...Done


No trigger
